In [0]:
# Imports
from pyspark.sql.functions import arrays_zip, col, explode, round, when

CATALOG = "optiridecatalog"

### Step 1 - Pick the batch  to process
Every extract

In [0]:
batch_id = spark.sql(f"SELECT max(batch_id) FROM {CATALOG}.bronze.bike_raw").collect()[0][0]
batch_id

## Step 2 . Look at what bronze actually holds

In [0]:
bike_bronze = spark.table(f"{CATALOG}.bronze.bike_raw").filter(col("batch_id") == batch_id)
print(f"Number of rows for the batch: {bike_bronze.count()}")
bike_bronze.printSchema()

In [0]:
bike_bronze.show(5)

## Step 3 Flatten the stations with explode

In [0]:
stations = bike_bronze.select(
    explode("network.stations").alias("s"),
    "batch_id",
    "ingest_at",
                              
)
print(f"Current Rows: {stations.count()}")
display(stations.limit(3))

## Step 4 Pick the fields and give them types

In [0]:
silver_stations = stations.selectExpr(
    "cast(s.id as string)                as station_id",
    "cast(s.name as string)              as station_name",
    "cast(s.latitude as double)          as latitude",
    "cast(s.longitude as double)         as longitude",
    "cast(s.name as string)              as address",
    "to_timestamp(regexp_replace(s.timestamp, 'Z$', ''))           as reading_ts",
    "cast(s.free_bikes as int)           as free_bikes",
    "cast(s.extra.slots as int)          as total_slots",
    "cast(s.extra.has_ebikes as boolean) as has_ebikes",
    "batch_id",
    "ingest_at",
)

display(silver_stations.limit(5))

## Step 5

In [0]:
silver_stations = (
    silver_stations
    .withColumn("available_slots", col("total_slots") - col("free_bikes"))
    .withColumn(
        "utilisation_rate",
        round(when(col("total_slots") > 0, col("free_bikes") / col("total_slots")), 3),
    )
)

display(silver_stations.select(
    "station_name", "free_bikes", "total_slots", "available_slots", "utilisation_rate"
).limit(5))

## Step 6

In [0]:
silver_stations = (
    silver_stations
    .filter(col("station_id").isNotNull() & col("reading_ts").isNotNull())
    .dropDuplicates(["station_id", "reading_ts"])
)

silver_stations.write.mode("append").option("mergeSchema", "true").saveAsTable(
    f"{CATALOG}.silver.station_status"
)

print(f"rows written to silver.station_status:, {silver_stations.count()}")

## Step 7

## Step 7 · The weather arrives in a different shape
Open-Meteo does not return a list of hours. It returns PARALLEL lists:
one list of times, one of temperatures, one of precipitation. Item 5 of
each list belongs to the same hour.
arrays_zip stitches those lists into one list of small records, and
explode then gives one row per hour. It is the Spark version of zip().

In [0]:
weather_bronze = spark.table(f"{CATALOG}.bronze.weather_raw").filter(col("batch_id") == batch_id)

hourly = weather_bronze.select(
    explode(
        arrays_zip(
            col("hourly.time").alias("time"),
            col("hourly.temperature_2m").alias("temperature_2m"),
            col("hourly.precipitation").alias("precipitation"),
            col("hourly.wind_speed_10m").alias("wind_speed_10m"),
            col("hourly.cloudcover").alias("cloudcover"),
            col("hourly.relativehumidity_2m").alias("relativehumidity_2m"),
        )
    ).alias("h"),                 # h = one hour
    "batch_id",
    "ingest_at",
)

print("hours after explode:", hourly.count())
display(hourly.limit(3))

## Step 8

## Step 8 · Type the weather and write it
Same pattern as the stations: name the columns, cast them, deduplicate
within the batch, append.
Note we deduplicate on weather_ts only WITHIN this batch. Across batches
the same hour appears again with a revised forecast, and keeping those
versions is deliberate. Gold decides what to do with them.

In [0]:
silver_weather = hourly.selectExpr(
    "to_timestamp(h.time)                  as weather_ts",
    "cast(h.temperature_2m as double)      as temperature_c",
    "cast(h.precipitation as double)       as precipitation_mm",
    "cast(h.wind_speed_10m as double)      as wind_speed_kmh",
    "cast(h.cloudcover as double)          as cloud_cover",
    "cast(h.relativehumidity_2m as double) as humidity",
    "batch_id",
    "ingest_at",
).dropDuplicates(["weather_ts"])

silver_weather.write.mode("append").option("mergeSchema", "true").saveAsTable(
    f"{CATALOG}.silver.weather_hourly"
)

print("rows written to silver.weather_hourly:", silver_weather.count())
display(silver_weather.orderBy("weather_ts").limit(5))

## Step 9 · Check before moving on
Never hand data to the next stage without asking whether it makes sense.
A failed notebook is cheap. A dashboard that has been quietly wrong for a
week is not.

In [0]:
display(spark.sql(f'''
    SELECT
        count(*)                                             AS rows,
        count(DISTINCT station_id)                           AS stations,
        min(reading_ts)                                      AS first_reading,
        max(reading_ts)                                      AS last_reading,
        sum(CASE WHEN free_bikes < 0 THEN 1 ELSE 0 END)      AS negative_bikes,
        sum(CASE WHEN total_slots IS NULL THEN 1 ELSE 0 END) AS missing_slots
    FROM {CATALOG}.silver.station_status
    WHERE batch_id = '{batch_id}'
'''))